<a href="https://colab.research.google.com/github/TienNguyen0712/mimic_iv/blob/main/notebooks/01_mimic_iv_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 **Mục tiêu**

Notebook này được viết nhằm thử nghiệm thiết kế một pipeline cho bộ dữ liệu MIMIC-IV giúp quản lý các bảng, làm sạch dữ liệu giám sát code cũng như chuẩn bi nguyên liệu cho các bước huấn luyện bài toán khác nhau

# 🏗️ **Kiến Trúc Công Nghệ**

Để tối ưu hóa hiệu năng xử lý trên môi trường cá nhân và Google Colab mà không cần đến cụm máy chủ cồng kềnh, hệ thống pipeline này lựa chọn một bộ công cụ tối giản nhưng cực kỳ mạnh mẽ:

| Thành phần | Công nghệ lựa chọn | Giải pháp thay thế (Production) |
|---|---|---|
| **Programming Language** | **Python 3.10+** | Python / Scala |
| **Data Processing Engine** | **DuckDB** | Apache Spark / Databricks |
| **Storage Format** | **Parquet (Apache)** | Delta Lake / Iceberg |
| **Configuration Layer** | **PyYAML (YAML Config)** | Hydra / Decouple |

---

# ❓**Lý Do Lựa Chọn, Ưu Điểm & Nhược Điểm**

## **Data Processing Engine: DuckDB**
> **DuckDB** được mệnh danh là "SQLite dành cho phân tích dữ liệu". Nó là một hệ quản trị cơ sở dữ liệu dạng cột (columnar) nhúng trực tiếp vào tiến trình Python, cho phép truy vấn SQL tốc độ cao trên các file dữ liệu lớn mà không cần cài đặt server độc lập.

*   **Tại sao lại sử dụng?** Bảng dữ liệu lâm sàng của MIMIC-IV (như `chartevents`) lên tới hơn 300 triệu dòng (nặng khoảng hơn 30GB thô). Nếu dùng Pandas, hệ thống sẽ cố nạp toàn bộ vào RAM dẫn đến sập nguồn do lỗi `Out of Memory`. DuckDB có khả năng xử lý stream qua ổ đĩa (Out-of-Core processing), giúp xử lý file 30GB mượt mà trên máy chỉ có 8GB RAM.
*   **Ưu điểm:**
    *   **Siêu nhẹ & Tiện lợi:** Cài đặt chỉ với `pip install duckdb`. Không cần cấu hình Java, Hadoop hay Docker cồng kềnh như Apache Spark.
    *   **Tốc độ vượt trội:** Thực thi các câu lệnh SQL (Filter, Aggregation) nhanh gấp hàng chục lần so với Pandas thuần nhờ kiến trúc Vectorized Execution.
    *   **Tích hợp sâu:** Đọc trực tiếp từ file CSV, Parquet và chuyển đổi qua lại với Pandas DataFrame hoặc PyArrow chỉ trong 1 dòng code.
*   **Nhược điểm:**
    *   **Hạn chế mở rộng đa máy (Horizontal Scalability):** DuckDB tối ưu hóa phần cứng trên một máy đơn lẻ (Single-node). Nếu dữ liệu tăng lên quy mô hàng trăm Terabyte/Petabyte trên cụm máy chủ Cloud, nó không thể thay thế cho Apache Spark.

## **Storage Format: Apache Parquet**
> **Parquet** là định dạng lưu trữ dữ liệu mã nguồn mở dạng cột (columnar storage), được thiết kế để tối ưu hóa việc lưu trữ và truy vấn trong hệ sinh thái Big Data.

*   **Tại sao lại sử dụng?** Dữ liệu MIMIC-IV được phân phối mặc định ở dạng file `.csv` thô. File CSV lưu dữ liệu theo từng dòng (row-oriented), khiến hệ thống phải quét qua toàn bộ các cột không liên quan khi truy vấn, gây lãng phí tài nguyên đĩa.
*   **Ưu điểm:**
    *   **Tỷ lệ nén cực cao:** Tách biệt dữ liệu theo cột giúp áp dụng các thuật toán nén (như Snappy) hiệu quả hơn. Dung lượng đĩa giảm từ 70% đến 80% so với CSV thô.
    *   **Kỹ thuật Predicate Pushdown:** Cho phép DuckDB đọc trực tiếp siêu dữ liệu (metadata) của file Parquet để bỏ qua các khối dữ liệu (Data Blocks) hoặc các phân vùng không thỏa mãn điều kiện `WHERE` trước khi nạp vào RAM.
*   **Nhược điểm:**
    *   **Không thể đọc bằng mắt thường:** Khác với CSV mở lên là thấy chữ, Parquet là file nhị phân (Binary), muốn xem nội dung bắt buộc phải dùng code hoặc công cụ chuyên dụng (như Parquet Viewer).
    *   **Chi phí ghi đè cao:** Việc cập nhật một vài dòng nhỏ lẻ (Insert/Update) trong file Parquet tốn chi phí hơn vì hệ thống phải ghi lại cả một block cột.

# 🥉 **Tầng 1: Bronze - Thực hiện việc phân vùng và lưu trữ các file thông qua DuckDB**

- Ý tưởng là thực hiện phân vùng chia nhỏ đối với các file có dung lượng lớn như file `chartevents`, `labelevents` chia theo chunking `subject_id` rồi sau đó chuyển đổi các file còn lại thành file parquet giúp cho việc load dễ dàng hơn

In [1]:
# Cài đặt thư viện $ Kết nối Drive
# Cập nhật DuckDB bản mới nhất
!pip install --upgrade duckdb -q

import duckdb
import os
import glob
import time
from google.colab import drive

# Mount Google Drive để đọc/ghi file
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 61.6 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
# Khởi tạo cấu trúc thư mục

# Định nghĩa các đường dẫn giả lập cấu trúc dự án
SOURCE_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2" # Đường dẫn file sample trên Drive
BRONZE_BASE_DIR = "./data/bronze"

# Tạo thư mục đầu ra trong môi trường Colab
os.makedirs(BRONZE_BASE_DIR, exist_ok=True)

In [3]:
# Định nghĩa các bảng lớn cần được Partition/Bucket để tránh quét toàn bộ ổ đĩa
# Các bảng khác nhỏ hơn sẽ được ghi thành 1 file duy nhất cho gọn
PARTITION_STRATEGY = {
    "chartevents": {"by": "subject_id", "buckets": 8},
    "labevents": {"by": "subject_id", "buckets": 8},
}

# Định nghĩa 7 bảng cần lấy để giảm tải hệ thống
SELECTED_TABLES = {
    "admissions",
    "icustays",
    "chartevents",
    "labevents",
    "inputevents",
    "outputevents",
    "procedureevents",
    "patients",
}

In [4]:
def dynamic_bronze_pipeline(source_dir, output_base_dir):
    # Khởi tạo DuckDB In-Memory Engine
    conn = duckdb.connect(database=':memory:')

    # Tìm tất cả các file .csv.gz nằm trong các thư mục con
    search_pattern = os.path.join(source_dir, "**/*.csv.gz")
    gz_files = glob.glob(search_pattern, recursive=True)

    if not gz_files:
        print("Không tìm thấy file .csv.gz nào trong thư mục nguồn!")
        return

    # Lọc lại danh sách file: Chỉ giữ những file nằm trong danh sách 7 bảng đã chọn
    filtered_files = []
    for file_path in gz_files:
        file_name = os.path.basename(file_path)
        table_name = file_name.replace(".csv.gz", "")
        if table_name in SELECTED_TABLES:
            filtered_files.append((file_path, table_name))

    if not filtered_files:
        print(f"Không tìm thấy file nào khớp với 8 bảng đã chọn trong thư mục nguồn!")
        return

    print(f"Tìm thấy {len(filtered_files)} / {len(gz_files)} file thuộc danh sách lựa chọn cần xử lý.\n" + "="*50)

    # Chạy vòng lặp trên các file đã được lọc
    for file_path, table_name in filtered_files:
        start_time = time.time()

        file_name = os.path.basename(file_path)
        module_name = os.path.basename(os.path.dirname(file_path))

        # Tạo đường dẫn đầu ra động theo cấu trúc: ./data/bronze/{module_name}/{table_name}
        target_dir = os.path.join(output_base_dir, module_name, table_name)
        os.makedirs(target_dir, exist_ok=True)

        print(f"Đang xử lý: [Module: {module_name}] -> [Bảng: {table_name}]")

        # Kiểm tra xem bảng này có cần áp dụng chiến lược Bucketing không
        if table_name in PARTITION_STRATEGY:
            strategy = PARTITION_STRATEGY[table_name]
            partition_column = strategy["by"]
            num_buckets = strategy["buckets"]

            # Sử dụng hàm hash() chuẩn của DuckDB
            query = f"""
                COPY (
                    SELECT *,
                           abs(hash({partition_column})) % {num_buckets} AS bucket
                    FROM read_csv_auto('{file_path}')
                )
                TO '{target_dir}'
                (FORMAT PARQUET, PARTITION_BY bucket, OVERWRITE_OR_IGNORE 1);
            """
            print(f"Áp dụng Bucketing theo cột: {partition_column} ({num_buckets} buckets)")
        else:
            # Với bảng nhỏ, ghi trực tiếp thành 1 file parquet duy nhất nằm trong thư mục đó
            output_file = os.path.join(target_dir, f"{table_name}.parquet")
            query = f"""
                COPY (SELECT * FROM read_csv_auto('{file_path}'))
                TO '{output_file}'
                (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
            """
            print(f"Ghi thành file đơn: {table_name}.parquet")

        # Thực thi câu lệnh qua DuckDB
        try:
            conn.execute(query)
            duration = time.time() - start_time
            print(f"Hoàn thành trong {duration:.2f} giây -> Lưu tại: {target_dir}\n" + "-"*40)
        except Exception as e:
            print(f"Lỗi khi xử lý file {file_name}: {str(e)}\n" + "-"*40)

In [5]:
dynamic_bronze_pipeline(SOURCE_DIR, BRONZE_BASE_DIR)

Tìm thấy 8 / 31 file thuộc danh sách lựa chọn cần xử lý.
Đang xử lý: [Module: icu] -> [Bảng: chartevents]
Áp dụng Bucketing theo cột: subject_id (8 buckets)
Hoàn thành trong 1.32 giây -> Lưu tại: ./data/bronze/icu/chartevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: icustays]
Ghi thành file đơn: icustays.parquet
Hoàn thành trong 0.38 giây -> Lưu tại: ./data/bronze/icu/icustays
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: inputevents]
Ghi thành file đơn: inputevents.parquet
Hoàn thành trong 0.55 giây -> Lưu tại: ./data/bronze/icu/inputevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: procedureevents]
Ghi thành file đơn: procedureevents.parquet
Hoàn thành trong 0.52 giây -> Lưu tại: ./data/bronze/icu/procedureevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: outputevents]
Ghi thành file đơn: outputevents.parquet
Hoàn thành trong 0.40 giây -> Lưu tại: ./d

# 🥈 **Tầng 2: Silver - Thực hiện việc làm sạch dữ liệu và Chuẩn hóa quan hệ**

Tàng này sẽ kiểm tra chất lượng của dữ liệu biến dữ liệu thô thành các dữ liệu có cấu trúc và đáng tin cậy.

Do đó cần giải quyết 4 bài toán sau:

1. **Ép kiểu dữ liệu (Data Type Casting):** File Parquet ở tầng Bronze được DuckDB tự động nhận diện kiểu dữ liệu `(read_csv_auto)`, nhưng đôi khi chưa tối ưu. Cần ép kiểu tường minh: các cột ID về kiểu INTEGER, các cột thời gian về kiểu TIMESTAMP, các chỉ số về kiểu FLOAT.

2. **Đồng bộ hóa thời gian (Datetime Standardization):** Dữ liệu y tế của MIMIC-IV tràn ngập các cột thời gian (admittime, dischtime, intime, outtime, charttime). Cần đồng bộ tất cả về một chuẩn chung và xử lý các lỗi logic (ví dụ: bộ lọc loại bỏ các dòng có ngày xuất viện trước ngày nhập viện).

3. **Xử lý giá trị khuyết (NULL handling):** Quyết định xem cột nào bắt buộc không được NULL (như các cột khóa chính `subject_id`, `hadm_id`), và cột nào nếu NULL thì xử lý mặc định (ví dụ: nếu kết quả xét nghiệm value là text bị khuyết thì thay bằng 'UNKNOWN').

4. **Tạo các liên kết thực thể (Referential Integrity):** Đảm bảo các bảng dữ liệu lâm sàng (`chartevents, labevents`) đều có thể kết nối (JOIN) mượt mà với bảng xương sống `admissions` và `patients`.

Tầng Silver sẽ được chia thành 2 giai đoạn:
- **Giai đoạn 1:** Dựng bảng trung tâm `patient_master`
- **Giai đoạn 2:** Xử lý 5 bảng sự kiện lâm sàng bằng kỹ thuật **Pivot dữ liệu** dựa them danh sách itemid của hệ thống mimic

In [6]:
# Định nghĩa phân chia thành các buckets nếu bảng nặng
PARTITION_STRATEGY = {
    "vital_signs": {"by": "stay_id", "buckets": 8},
    "laboratory": {"by": "stay_id", "buckets": 8},
}

def get_duckdb_connection():
    """Khởi tạo kết nối DuckDB In-Memory."""
    return duckdb.connect(database=':memory:')

## **Giai đoạn 1: Xây dựng bảng Trung Tâm `patient_master`**

Bảng này đóng vai trò là "Xương sống" (Master Anchor). Tất cả các bảng lâm sàng phía sau khi làm sạch đều phải INNER JOIN với bảng này theo stay_id để lọc bỏ những dữ liệu nằm ngoài thời gian nằm viện hoặc sai lệch định danh.

Các bước làm sạch & chuẩn hóa:
1. **Đồng bộ Khóa:** Kết hợp `admissions` và `icustays` qua `subject_id` và `hadm_id`.
2. **Chuẩn hóa Thời gian:** Ép kiểu toàn bộ các cột thời gian về dạng `TIMESTAMP`.
3. Xử lý logic thời gian bất thường: Loại bỏ các bản ghi có thời gian ra viện trước thời gian vào viện (`icu_outtime` < `icu_intime` hoặc `discharge_time` < `admission_time`).

In [7]:
def build_patient_master(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 1: Đọc bảng admissions và icustays từ tầng Bronze,
    làm sạch logic thời gian và sinh bảng patient_master ở tầng Silver.
    """
    print("=== [GIAI ĐOẠN 1] Bắt đầu xây dựng 'patient_master' ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # 1. Khai báo và kiểm tra tệp nguồn Bronze
    required_tables = ["admissions", "icustays"]
    for table in required_tables:
        bronze_path = glob.glob(os.path.join(bronze_dir, f"**/{table}.parquet"), recursive=True) or \
                      glob.glob(os.path.join(bronze_dir, f"**/{table}"), recursive=True)
        if not bronze_path:
            raise FileNotFoundError(f"[X] Không tìm thấy dữ liệu Bronze cho bảng bắt buộc: {table}")

        path_to_read = bronze_path[0]
        # NẾU là thư mục (chứa các bucket), ta quét toàn bộ file parquet bên trong bằng `/**/*.parquet`
        if os.path.isdir(path_to_read):
            query_read = f"SELECT * FROM read_parquet('{path_to_read}/**/*.parquet', hive_partitioning=1)"
        # NẾU chỉ là một file đơn lẻ, ta đọc trực tiếp file đó
        else:
            query_read = f"SELECT * FROM read_parquet('{path_to_read}')"

        conn.execute(f"CREATE OR REPLACE VIEW bronze_{table} AS {query_read}")
        print(f" -> Đã kết nối thành công: bronze_{table}")

    # 2. Định nghĩa thư mục đầu ra
    master_target_dir = os.path.join(silver_dir, "core", "patient_master")
    os.makedirs(master_target_dir, exist_ok=True)

    # 3. Thực thi SQL ETL làm sạch hành chính
    query_master = f"""
        COPY (
            WITH ranked_icu AS (
                SELECT
                    i.stay_id,      -- Mã lần nhập viện, Mã lần nằm viện, Mã bệnh nhân
                    i.hadm_id,
                    i.subject_id,
                    CAST(a.admittime AS TIMESTAMP) AS admission_time, -- Thời gian làm thủ tục
                    CAST(a.dischtime AS TIMESTAMP) AS discharge_time, -- Thời gian hoàn thành thủ tục
                    CAST(i.intime AS TIMESTAMP) AS icu_intime, -- Thời gian vào ICU
                    CAST(i.outtime AS TIMESTAMP) AS icu_outtime, -- Thời gian ra ICU
                    a.admission_type, -- Kiểu nhập viện
                    i.first_careunit AS careunit, -- Đơn vị chăm sóc đầu tiên
                    CAST(a.deathtime AS TIMESTAMP) AS deathtime, -- Thời gian tử vong trong viện
                    i.los AS los_days,
                    ROW_NUMBER() OVER (PARTITION BY i.hadm_id ORDER BY CAST(i.intime AS TIMESTAMP)) as rnk
                FROM bronze_icustays i
                INNER JOIN bronze_admissions a
                   ON i.hadm_id = a.hadm_id AND i.subject_id = a.subject_id
                WHERE i.stay_id IS NOT NULL
            )
            SELECT
                stay_id, hadm_id, subject_id, admission_time, discharge_time,
                icu_intime, icu_outtime, admission_type, careunit, deathtime, los_days
            FROM ranked_icu
            WHERE rnk = 1               -- Quy tắc 1: Chỉ giữ đợt nằm ICU đầu tiên
              AND los_days >= 1.0       -- Quy tắc 2: Thời gian nằm ICU tối thiểu 24 giờ (>= 1 ngày)
              AND icu_intime <= icu_outtime
        ) TO '{os.path.join(master_target_dir, "patient_master.parquet")}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """
    try:
        conn.execute(query_master)
        duration = time.time() - start_time
        print(f" -> [THÀNH CÔNG] Đã lưu 'patient_master' tại: {master_target_dir} ({duration:.2f} giây)\n")
    except Exception as e:
        print(f" -> [THẤT BẠI] Lỗi Giai đoạn 1: {str(e)}")
        raise e

Bên cạnh đó để phục vụ cho các bước sau ta cũng sẽ cần cột của các bảng
- `patients`:  Như tuổi, giới tính
- `chartevents`: Như cân nặng và chiều cao

Do đó ta sẽ có thêm một bảng `patients_clean` sinh ra nhằm JOIN với bảng trên

In [8]:
def build_patients_clean(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 1.1:
    Làm sạch bảng patients từ Bronze và sinh bảng patients_clean ở Silver.
    """

    print("=== [GIAI ĐOẠN 1.1] Bắt đầu xây dựng 'patients_clean' ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # 1. Khai báo và kiểm tra tệp nguồn Bronze
    bronze_path = glob.glob(os.path.join(bronze_dir, "**/patients.parquet"), recursive=True) or \
                  glob.glob(os.path.join(bronze_dir, "**/patients"), recursive=True)
    if not bronze_path:
        raise FileNotFoundError("[X] Không tìm thấy bảng patients")

    path_to_read = bronze_path[0]
    if os.path.isdir(path_to_read):
        query_read = f"SELECT * FROM read_parquet('{path_to_read}/**/*.parquet', hive_partitioning=1)"
    else:
        query_read = f"SELECT * FROM read_parquet('{path_to_read}')"
    conn.execute(f"""
        CREATE OR REPLACE VIEW bronze_patients AS
        {query_read}
    """)

    # 2. Định nghĩa thư mục đầu ra
    target_dir = os.path.join(silver_dir, "core", "patients_clean")
    os.makedirs(target_dir, exist_ok=True)

    # 3. Lệnh SQL
    query = f"""
    COPY (
        SELECT DISTINCT
            subject_id,
            gender,
            CAST(anchor_age AS INTEGER)  AS anchor_age,
            CAST(anchor_year AS INTEGER) AS anchor_year
        FROM bronze_patients
        WHERE subject_id IS NOT NULL
    )
    TO '{os.path.join(target_dir,"patients_clean.parquet")}'
    (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """

    try:
        conn.execute(query)
        print(f" -> [THÀNH CÔNG] patients_clean ({time.time()-start_time:.2f} giây)\n")
    except Exception as e:
        print(f" -> [THẤT BẠI] Lỗi Giai đoạn 1.1: {str(e)}")
        raise e

In [13]:
def build_height_weight(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 1.2:
    Trích xuất, quy đổi đơn vị và làm sạch dữ liệu Chiều cao (Height) & Cân nặng (Weight) từ chartevents.
    """
    print("=== [GIAI ĐOẠN 1.2] Bắt đầu xây dựng 'height_weight' ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # Tìm đường dẫn chartevents (ưu tiên .parquet, sau đó là dạng thư mục)
    bronze_path = (
        glob.glob(os.path.join(bronze_dir, "**/chartevents.parquet"), recursive=True) or
        glob.glob(os.path.join(bronze_dir, "**/chartevents"), recursive=True)
    )
    if not bronze_path:
        raise FileNotFoundError("[X] Không tìm thấy bảng chartevents")

    path_to_read = bronze_path[0]

    # Rút gọn logic đọc file: Nếu là thư mục thì thêm '/**/*.parquet', nếu là file thì giữ nguyên
    read_pattern = f"{path_to_read}/**/*.parquet" if os.path.isdir(path_to_read) else path_to_read

    # Tạo VIEW trực tiếp từ đường dẫn đã xử lý (sử dụng hive_partitioning phòng trường hợp dữ liệu phân mảnh)
    conn.execute(f"""
        CREATE OR REPLACE VIEW bronze_chartevents AS
        SELECT * FROM read_parquet('{read_pattern}', hive_partitioning=1)
    """)

    target_file = os.path.join(silver_dir, "core", "height_weight", "height_weight.parquet")
    os.makedirs(os.path.dirname(target_file), exist_ok=True)

    # Rút gọn SQL: Gộp trực tiếp câu lệnh COPY và làm sạch CTE
    query = f"""
    COPY (
            WITH raw_metrics AS (
                SELECT
                    stay_id,
                    CASE
                        WHEN itemid = 226730 THEN valuenum                          -- Chiều cao dạng cm
                        WHEN itemid = 226707 THEN valuenum * 2.54                   -- Chiều cao dạng inch -> đổi sang cm
                        ELSE NULL
                    END AS height_derived,
                    CASE
                        WHEN itemid IN (224639, 226512) THEN valuenum               -- Cân nặng dạng kg
                        WHEN itemid = 226531 THEN valuenum * 0.45359237             -- Cân nặng dạng lbs -> đổi sang kg
                        ELSE NULL
                    END AS weight_derived
                FROM bronze_chartevents
                WHERE valuenum IS NOT NULL
                  AND itemid IN (226730, 226707, 224639, 226512, 226531)
                  AND stay_id IS NOT NULL
            )
            SELECT
                stay_id,
                -- Lọc nhiễu lâm sàng: Chiều cao người trưởng thành từ 50cm đến 250cm
                ROUND(AVG(CASE WHEN height_derived BETWEEN 50 AND 250 THEN height_derived END), 2) AS height_cm,
                -- Lọc nhiễu lâm sàng: Cân nặng từ 10kg đến 300kg
                ROUND(AVG(CASE WHEN weight_derived BETWEEN 10 AND 300 THEN weight_derived END), 2) AS weight_kg
            FROM raw_metrics
            GROUP BY stay_id
            HAVING height_cm IS NOT NULL OR weight_kg IS NOT NULL
        ) TO '{target_file}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """
    conn.execute(query)
    print(f" -> [THÀNH CÔNG] height_weight ({time.time() - start_time:.2f}s)\n")

## **Giai đoạn 2: Xử lý các bảng sự kiện lâm sàng**

**Nhóm Bảng Lâm Sàng Phân Tích (Vital Signs & Laboratory)**

Đối với `chartevents` và `labevents`, đặc trưng của dữ liệu MIMIC là lưu theo dạng EAV (Entity-Attribute-Value) – mỗi chỉ số là một dòng. Ở tầng Silver, ta cần Mapping `itemid` sang tên đặc trưng và lọc nhiễu lâm sàng (Outliers).

- Bảng vital_signs (Từ chartevents)
Mapping itemid chuẩn (MIMIC-IV):

  - Heart Rate: `220045` | Respiratory Rate `220210`
  - SysBP: `220179, 220050` | DiasBP: `220180, 220051`
  - MeanBP: `220052, 220181`
  - RespRate: `220210` | Temperature: `223761` (°C), `223762` (°F)
  - SpO2: `220277` | FiO2: `223835`

  **Chuẩn hóa đơn vị:** Chuyển đổi Nhiệt độ từ Fahrenheit sang Celsius: $C = (F - 32) * 5/9$.

- Bảng laboratory (Từ `labevents`)
  - Mapping `itemid` chuẩn: Creatinine `(50912)`, Lactate `(50813)`, WBC `(51301)`, Hemoglobin `(51222)`, Platelet `(51265)`, AST `(50861)`, ALT `(50863)`, Sodium `(50983)`, Potassium `(50971)`.
  - **Lưu ý:** labevents gốc chỉ có subject_id và hadm_id. Ta phải join với patient_master để lấy ra đúng stay_id dựa vào thời gian làm xét nghiệm charttime.

**Nhóm Bảng Can Thiệp Điều Trị (Medication, Outputs, Procedures)**

- Bảng medication (Từ inputevents - Thuốc truyền tĩnh mạch/vận mạch)
  - Làm sạch đặc thù: Tính toán tổng liều lượng hoặc giữ nguyên tốc độ truyền (rate), chuẩn hóa thời gian bắt đầu (starttime) và kết thúc (endtime).

  - Itemid gợi ý: Vasopressor (Norepinephrine: 221906), IV Fluid (NSS: 225158), Insulin (225152).

- Bảng outputs (Từ outputevents - Lượng dịch ra)
  - Làm sạch đặc thù: Gom cụm lượng nước tiểu hoặc dịch dẫn lưu theo từng giờ hoặc giữ nguyên mốc thời gian để tính Balance dịch.

- Bảng procedures (Từ procedureevents - Thủ thuật lâm sàng)
    - Làm sạch đặc thù: Tính toán thời gian can thiệp (duration_hours = endtime - starttime). Khai báo trạng thái thủ thuật (Ví dụ: Đang thở máy = 1, Không thở máy = 0)

In [18]:
def process_clinical_events(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 2: Đọc bảng patient_master từ Silver và 5 bảng sự kiện từ Bronze.
    Tiến hành Pivot, chuẩn hóa đơn vị, lọc nhiễu và lưu trữ (có Bucketing).
    """
    print("=== [GIAI ĐOẠN 2] Bắt đầu xử lý các bảng sự kiện lâm sàng ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # 1. Đăng ký liên kết tới bảng Master vừa tạo từ Giai đoạn 1
    master_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")
    if not glob.glob(master_path):
        raise FileNotFoundError("[X] Không tìm thấy bảng patient_master của Giai đoạn 1. Vui lòng chạy Giai đoạn 1 trước!")
    conn.execute(f"CREATE OR REPLACE VIEW silver_patient_master AS SELECT * FROM read_parquet('{master_path}')")

    # 2. Đăng ký các bảng lâm sàng còn lại từ tầng Bronze
    clinical_source_tables = ["chartevents", "labevents", "inputevents", "outputevents", "procedureevents"]
    for table in clinical_source_tables:
        bronze_path = glob.glob(os.path.join(bronze_dir, f"**/{table}.parquet"), recursive=True) or \
                      glob.glob(os.path.join(bronze_dir, f"**/{table}"), recursive=True)
        if bronze_path:
            path_to_read = bronze_path[0]
            if os.path.isdir(path_to_read):
                query_read = f"SELECT * FROM read_parquet('{path_to_read}/**/*.parquet', hive_partitioning=1)"
            else:
                query_read = f"SELECT * FROM read_parquet('{path_to_read}')"

            conn.execute(f"CREATE OR REPLACE VIEW bronze_{table} AS {query_read}")
            print(f" -> Đã kết nối thành công: bronze_{table}\n")

    # 3. Từ điển chứa logic SQL biến đổi chuyên sâu (đồng bộ tên cột thời gian về 'charttime')
    queries_clinical_tables = {
        "vital_signs": """
            SELECT
                p.stay_id,
                CAST(c.charttime AS TIMESTAMP) AS charttime,
                CASE WHEN c.itemid = 220045 AND c.valuenum BETWEEN 30 AND 250 THEN c.valuenum END AS heart_rate,
                CASE WHEN c.itemid IN (220179, 220050) AND c.valuenum BETWEEN 40 AND 280 THEN c.valuenum END AS sys_bp,
                CASE WHEN c.itemid IN (220180, 220051) AND c.valuenum BETWEEN 20 AND 180 THEN c.valuenum END AS dias_bp,
                CASE WHEN c.itemid IN (220052, 220181, 224322) AND c.valuenum BETWEEN 20 AND 200 THEN c.valuenum END AS map_bp,
                CASE WHEN c.itemid = 220210 AND c.valuenum BETWEEN 4 AND 60 THEN c.valuenum END AS resp_rate,
                CASE
                    -- 223761 là Fahrenheit (86-113 F), 223762 là Celsius (30-45 C)
                    WHEN c.itemid = 223762 AND c.valuenum BETWEEN 30 AND 45 THEN c.valuenum
                    WHEN c.itemid = 223761 AND c.valuenum BETWEEN 86 AND 113 THEN (c.valuenum - 32) * 5 / 9
                END AS temperature,
                CASE WHEN c.itemid = 220277 AND c.valuenum BETWEEN 50 AND 100 THEN c.valuenum END AS spo2
            FROM bronze_chartevents c
            INNER JOIN silver_patient_master p ON c.stay_id = p.stay_id
            WHERE CAST(c.charttime AS TIMESTAMP) BETWEEN p.icu_intime AND p.icu_outtime
              AND c.itemid IN (220045, 220179, 220050, 220180, 220051, 220052, 220181, 224322, 220210, 223761, 223762, 220277)
        """,

        "laboratory": """
            SELECT
                p.stay_id,
                CAST(l.charttime AS TIMESTAMP) AS charttime,
                AVG(CASE WHEN l.itemid = 50912 THEN l.valuenum END) AS creatinine,
                AVG(CASE WHEN l.itemid = 50813 THEN l.valuenum END) AS lactate,
                AVG(CASE WHEN l.itemid = 51301 THEN l.valuenum END) AS white_blood_cell,
                AVG(CASE WHEN l.itemid = 51222 THEN l.valuenum END) AS hemoglobin,
                AVG(CASE WHEN l.itemid = 51265 THEN l.valuenum END) AS platelet,
                AVG(CASE WHEN l.itemid = 50983 THEN l.valuenum END) AS sodium,
                AVG(CASE WHEN l.itemid = 50971 THEN l.valuenum END) AS potassium
            FROM bronze_labevents l
            INNER JOIN silver_patient_master p ON l.hadm_id = p.hadm_id AND l.subject_id = p.subject_id
            WHERE CAST(l.charttime AS TIMESTAMP) BETWEEN p.icu_intime AND p.icu_outtime
              AND l.itemid IN (50912, 50813, 51301, 51222, 51265, 50983, 50971)
            GROUP BY p.stay_id, l.charttime
        """,

        "medication": """
            SELECT
                p.stay_id,
                CAST(i.starttime AS TIMESTAMP) AS starttime,
                CAST(i.endtime AS TIMESTAMP) AS endtime,
                CASE
                    WHEN i.itemid = 221906 THEN 'Norepinephrine'
                    WHEN i.itemid = 225152 THEN 'Insulin'
                    ELSE 'Other Med'
                END AS med_name,
                i.itemid,
                i.amount AS dose,
                i.rate AS rate
            FROM bronze_inputevents i
            INNER JOIN silver_patient_master p ON i.stay_id = p.stay_id
            WHERE i.statusdescription != 'Rewritten'
              AND CAST(i.starttime AS TIMESTAMP) BETWEEN p.icu_intime AND p.icu_outtime
        """,

        "outputs": """
            SELECT
                p.stay_id,
                CAST(o.charttime AS TIMESTAMP) AS charttime,
                o.itemid,
                CASE WHEN o.itemid IN (226559, 226560, 226561, 227488) THEN 'Urine Output' ELSE 'Other Output' END AS output_type,
                o.value AS volume_ml
            FROM bronze_outputevents o
            INNER JOIN silver_patient_master p ON o.stay_id = p.stay_id
            WHERE o.value BETWEEN 0 AND 5000
              AND CAST(o.charttime AS TIMESTAMP) BETWEEN p.icu_intime AND p.icu_outtime
        """,

        "procedures": """
            SELECT
                p.stay_id,
                CAST(pr.starttime AS TIMESTAMP) AS starttime,
                CAST(pr.endtime AS TIMESTAMP) AS endtime,
                pr.itemid,
                CASE WHEN pr.itemid = 227194 THEN 'Mechanical Ventilation' WHEN pr.itemid = 225792 THEN 'ECMO' ELSE 'Other' END AS procedure_name,
                date_diff('minute', CAST(pr.starttime AS TIMESTAMP), CAST(pr.endtime AS TIMESTAMP)) / 60.0 AS duration_hours
            FROM bronze_procedureevents pr
            INNER JOIN silver_patient_master p ON pr.stay_id = p.stay_id
            WHERE pr.statusdescription != 'Annull'
              AND CAST(pr.starttime AS TIMESTAMP) BETWEEN p.icu_intime AND p.icu_outtime
        """
    }

    # 4. Chạy vòng lặp xử lý và lưu trữ dữ liệu
    for target_table, core_sql in queries_clinical_tables.items():
        table_start = time.time()
        target_dir = os.path.join(silver_dir, "clinical", target_table)
        os.makedirs(target_dir, exist_ok=True)

        # Áp dụng chiến lược phân tách dữ liệu (Bucketing 8 files) bằng định dạng cột chuẩn 'bucket_id'
        if target_table in PARTITION_STRATEGY:
            strategy = PARTITION_STRATEGY[target_table]
            bucket_column = strategy["by"]
            num_buckets = strategy["buckets"]

            export_query = f"""
                COPY (
                    SELECT *, abs(hash({bucket_column})) % {num_buckets} AS bucket
                    FROM ({core_sql})
                ) TO '{target_dir}' (FORMAT PARQUET, PARTITION_BY bucket, OVERWRITE_OR_IGNORE 1);
            """
            print(f" -> Đang xử lý: {target_table} (Chia làm {num_buckets} Buckets theo {bucket_column} -> bucket)")
        else:
            export_query = f"""
                COPY ({core_sql}) TO '{os.path.join(target_dir, f"{target_table}.parquet")}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
            """
            print(f" -> Đang xử lý: {target_table}")

        try:
            conn.execute(export_query)
            print(f"   -> Hoàn thành bảng {target_table} trong {time.time() - table_start:.2f} giây.\n")
        except Exception as e:
            print(f"   [X] LỖI tại bảng {target_table}: {str(e)}")

    print(f"\n -> [THÀNH CÔNG] Toàn bộ Giai đoạn 2 hoàn tất trong {time.time() - start_time:.2f} giây.\n")

In [19]:
if __name__ == "__main__":
    BRONZE_DIR = "./data/bronze"
    SILVER_DIR = "./data/silver"

    print("KHỞI ĐỘNG HỆ THỐNG PIPELINE TẦNG SILVER\n" + "="*50)

    # Bước 1: Sinh bảng trung tâm trước
    build_patient_master(BRONZE_DIR, SILVER_DIR)
    build_patients_clean(BRONZE_DIR, SILVER_DIR)
    build_height_weight(BRONZE_DIR, SILVER_DIR)

    # Bước 2: Dựa vào bảng trung tâm để xử lý sạch tất cả các dữ liệu lâm sàng
    process_clinical_events(BRONZE_DIR, SILVER_DIR)

    print("="*50 + "\n[DONE] Pipeline đã hoàn thành độc lập cả 2 giai đoạn.")

KHỞI ĐỘNG HỆ THỐNG PIPELINE TẦNG SILVER
=== [GIAI ĐOẠN 1] Bắt đầu xây dựng 'patient_master' ===
 -> Đã kết nối thành công: bronze_admissions
 -> Đã kết nối thành công: bronze_icustays
 -> [THÀNH CÔNG] Đã lưu 'patient_master' tại: ./data/silver/core/patient_master (0.03 giây)

=== [GIAI ĐOẠN 1.1] Bắt đầu xây dựng 'patients_clean' ===
 -> [THÀNH CÔNG] patients_clean (0.01 giây)

=== [GIAI ĐOẠN 1.2] Bắt đầu xây dựng 'height_weight' ===
 -> [THÀNH CÔNG] height_weight (0.03s)

=== [GIAI ĐOẠN 2] Bắt đầu xử lý các bảng sự kiện lâm sàng ===
 -> Đã kết nối thành công: bronze_chartevents

 -> Đã kết nối thành công: bronze_labevents

 -> Đã kết nối thành công: bronze_inputevents

 -> Đã kết nối thành công: bronze_outputevents

 -> Đã kết nối thành công: bronze_procedureevents

 -> Đang xử lý: vital_signs (Chia làm 8 Buckets theo stay_id -> bucket)
   -> Hoàn thành bảng vital_signs trong 0.07 giây.

 -> Đang xử lý: laboratory (Chia làm 8 Buckets theo stay_id -> bucket)
   -> Hoàn thành bảng labora

# 🥇 **Tầng 3: Gold - Thực hiện việc chuẩn bị dữ liệu cho các bài toán đầu ra**

Khác với mô hình Data Mart truyền thống phục vụ báo cáo BI, tầng Gold được thiết kế theo mô hình Feature Store chuyên dụng cho AI/ML. Mỗi bản ghi trong Feature Store đại diện cho một thực thể lâm sàng duy nhất (`stay_id`), nơi toàn bộ các đặc trưng động và tĩnh được tính toán sẵn trên nhiều cửa sổ thời gian khác nhau để tạo thành một Ma trận Đặc trưng (Feature Matrix) phẳng, sẵn sàng cho việc huấn luyện mô hình.

Để tối ưu hóa chi phí I/O (đọc/ghi đĩa) và giải phóng tài nguyên bộ nhớ khi huấn luyện, tầng Gold thiết lập cơ chế hợp nhất vật lý (Physical Flattening). Thay vì lưu thành nhiều bảng rời rạc gây tốn tài nguyên `JOIN` lúc Train, toàn bộ các nhóm tính năng được xử lý song song dưới dạng các khối logic (CTE) rồi gom về một tập dữ liệu phẳng duy nhất được phân mảnh (Bucketing 8 files) hiệu năng cao.

## **Kiến trúc các Feature Groups thành phần**
Mặc dù được lưu trữ tập trung dưới một Ma trận Đặc trưng duy nhất, hệ thống Feature Store vẫn đảm bảo tính tường minh và quản lý chặt chẽ theo 5 nhóm tính năng logic (Feature Groups) cốt lõi:

- **Static Features (Đặc trưng thể trạng & hành chính):** Thu thập các thông tin cố định hoặc ít biến động tại thời điểm nhập viện như Giới tính (gender), Tuổi thực tế khi vào ICU (đã hiệu chỉnh chênh lệch năm bảo mật) và chỉ số khối cơ thể BMI (tính toán tự động từ Chiều cao & Cân nặng đã làm sạch ở tầng Silver).

- **Vital Signs Features (Đặc trưng dấu hiệu sinh tồn đa cửa sổ):** Tổng hợp động các chỉ số huyết động và hô hấp gồm Nhịp tim (heart_rate), Huyết áp tâm thu (sys_bp), Huyết áp tâm trương (dias_bp), Huyết áp trung bình (map_bp), Nhịp thở (resp_rate), Nhiệt độ cơ thể (temperature) và Độ bão hòa oxy (spo2).

- **Laboratory Features (Đặc trưng cận lâm sàng đa cửa sổ):** Tính toán giá trị trung bình/lớn nhất của các chỉ số xét nghiệm máu quan trọng như creatinine, lactate, Bạch cầu (white_blood_cell), hemoglobin và Tiểu cầu (platelet).

- **Medication Features (Đặc trưng can thiệp dược lý):** Định lượng tổng liều lượng tích lũy của các nhóm thuốc đặc hiệu cao như thuốc vận mạch (Norepinephrine), Insulin phục vụ các bài toán tiên lượng suy đa tạng.

- **Fluid Outputs Features (Đặc trưng dịch ra):** Theo dõi sát tổng thể tích nước tiểu (Urine Output) tích lũy theo từng mốc thời gian để làm tiền đề đánh giá chức năng thận.

### **Kỹ thuật Cửa sổ Thời gian (Multi-Window) & Chống rò rỉ dữ liệu**

- **Tính toán Thời gian Động (Temporal Aggregation):** Thay vì làm tròn thời gian cố định theo block (như date_trunc 4 giờ truyền thống), kỹ thuật Feature Store sử dụng hàm tính khoảng cách thời gian chính xác date_diff('hour', icu_intime, charttime) để xác định một sự kiện lâm sàng thuộc về cửa sổ nào ($1\text{h}, 6\text{h}, 12\text{h}, 24\text{h}$) tính từ mốc $T_0$ (thời điểm nhập khoa ICU).
- **Nghiêm ngặt chống rò rỉ dữ liệu (Anti-Data Leakage):** Việc áp dụng kỹ thuật lũy tiến qua điều kiện BETWEEN 0 AND X giờ giúp đóng băng dữ liệu tại các mốc thời gian cố định. Điều này đảm bảo mô hình AI chỉ học các thông tin trong quá khứ/hiện tại của cửa sổ đó để dự báo biến cố tương lai, loại bỏ hoàn toàn rủi ro rò rỉ dữ liệu của cả đợt điều trị vào mô hình.

### **Kiến trúc "Một Feature Store — Nhiều Nhãn Đầu Ra" (One-to-Many Labels Mapping)**

Cấu trúc phẳng hóa này cho phép tách biệt hoàn toàn phần Tính năng (Features) và phần Nhãn (Labels) về mặt lưu trữ vật lý. Tập dữ liệu đặc trưng nằm trong thư mục features/ sau khi xuất xưởng sẽ đóng vai trò là một "Core Engine" bất biến, có thể tái sử dụng ngay lập tức cho nhiều bài toán Machine Learning khác nhau bằng cách liên kết `(LEFT JOIN)` trực tiếp với các bảng nhãn mục tiêu được lưu độc lập tại thư mục `labels/`:
- **Dự đoán Tử vong:** `LEFT JOIN labels/mortality_labels.parquet ON stay_id`(Xác định tử vong nội viện/trong ICU).
- **Dự đoán Suy thận cấp:** `LEFT JOIN labels/aki_labels.parquet ON stay_id`(Phân độ suy thận cấp từ 0-3 theo tiêu chuẩn lâm sàng KDIGO dựa trên Creatinine và Nước tiểu).
- **Dự đoán Nhiễm trùng huyết:** L`EFT JOIN labels/sepsis_labels.parquet ON stay_id` (Gán nhãn biến cố Sepsis-3 thông qua proxy lâm sàng sử dụng thuốc vận mạch).
- **Dự đoán Thời gian nằm viện kéo dài:** `LEFT JOIN labels/los_labels.parquet ON stay_id` (Phân loại bệnh nhân có nguy cơ quá tải hạ tầng ICU với mốc $> 3$ ngày).

**Hiệu quả tối ưu:** Nhờ kiến trúc này, các kỹ sư Data Science có thể tự do thử nghiệm, thay đổi định nghĩa bài toán hoặc chuyển đổi giữa các mô hình học máy khác nhau mà hoàn toàn không cần phải viết lại hay chạy lại các pipeline xử lý dữ liệu thô cực nhọc và tốn tài nguyên.

In [20]:
SILVER_BASE_DIR = "./data/silver"
GOLD_BASE_DIR = "./data/gold"

# Khởi tạo DuckDB In-Memory Engine
conn = duckdb.connect(database=':memory:')

def print_group_summary(group_name, table_name):
    """Hàm hiển thị kết quả trực quan cho từng Feature Group"""
    res = conn.execute(f"SELECT COUNT(*), COUNT(DISTINCT stay_id) FROM {table_name}").fetchone()
    cols = conn.execute(f"PRAGMA table_info('{table_name}')").fetchall()
    col_names = [c[1] for c in cols]

    print(f"\n[FEATURE GROUP] {group_name}:")
    print(f"  - Bảng bộ nhớ   : {table_name}")
    print(f"  - Kích thước    : {res[0]} dòng x {len(cols)} cột")
    print(f"  - ICU Stays     : {res[1]}")
    print(f"  - Các cột chính : {', '.join(col_names[:6])} ...")
    print("-" * 60)

## **1. Patient Static Features (`patient_static_features`)**
**Mô tả:** Nhóm đặc trưng nhân khẩu học, hành chính và thể trạng cố định của bệnh nhân tại thời điểm nhập viện khoa Hồi sức tích cực. Đây là nhóm tính năng nền tảng giúp mô hình AI phân loại phân khúc bệnh nhân và hiệu chỉnh các yếu tố nhiễu sinh học.

**Các tính năng tính toán:**
  - **Tuổi thực tế khi vào ICU ($Age$):** Tính toán chính xác bằng công thức hiệu chỉnh khoảng cách năm giữa thời điểm nhập viện (icu_intime) và năm mốc bảo mật (anchor_year), thay vì dùng trực tiếp tuổi thô nhằm đảm bảo tính chính xác trong môi trường lâm sàng.
  - **Giới tính ($Gender$):** Mã hóa thông tin giới tính sinh học của bệnh nhân.Chiều cao
  - **($Height$) & Cân nặng ($Weight$):** Lấy từ dữ liệu thể trạng đã được làm sạch, quy đổi đồng bộ về hệ Metric (cm và kg) và loại bỏ các Outliers nhiễu lâm sàng ở tầng Silver.
  - **Chỉ số khối cơ thể ($BMI$):** Tính toán tự động thông qua công thức toán học phẳng:$$BMI = \frac{Weight_{kg}}{\left(\frac{Height_{cm}}{100}\right)^2}- $$
  - **Loại hình nhập viện ($Admission\ Type$) & Loại hình khoa ICU ($ICU\ Type$):** Các đặc trưng hành chính giúp mô hình nắm bắt được tuyến bệnh nhân (Cấp cứu, chuyển viện, mổ chương trình) và chuyên khoa tiếp nhận (MICU, SICU, CCU, CVICU...).

In [22]:
def build_feature_group_1(silver_dir, conn=None):
    """
    Feature Group 1:
    Xây dựng bảng patient_static_features từ các bảng Silver.
    """
    print(" -> [FG1] Đang xây dựng: patient_static_features...")

    # Đảm bảo có kết nối DuckDB
    if conn is None:
        conn = get_duckdb_connection()

    # Định nghĩa đường dẫn ngắn gọn tới các bảng nguồn tầng Silver
    pm_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")
    p_path = os.path.join(silver_dir, "core", "patients_clean", "*.parquet")
    hw_path = os.path.join(silver_dir, "core", "height_weight", "*.parquet")

    # Kiểm tra sự tồn tại của dữ liệu trước khi truy vấn
    for path_check, name in zip([pm_path, p_path, hw_path], ["patient_master", "patients_clean", "height_weight"]):
        if not glob.glob(path_check):
            raise FileNotFoundError(f"[X] Không tìm thấy dữ liệu Silver cho bảng: {name} tại {path_check}")

    query = f"""
    CREATE OR REPLACE TABLE patient_static_features AS
    SELECT
        pm.stay_id,
        pm.subject_id,
        pm.hadm_id,

        -- Demographic (Tính toán tuổi thực tế tại thời điểm vào ICU)
        (p.anchor_age + (EXTRACT(YEAR FROM CAST(pm.icu_intime AS TIMESTAMP)) - p.anchor_year)) AS age,
        p.gender,

        -- Anthropometric (Thể trạng đã làm sạch & Tính toán BMI)
        hw.height_cm,
        hw.weight_kg,
        CASE
            WHEN hw.height_cm > 0 AND hw.weight_kg IS NOT NULL
            THEN ROUND(hw.weight_kg / POWER(hw.height_cm / 100.0, 2), 2)
            ELSE NULL
        END AS bmi,

        -- Administrative
        pm.admission_type,
        pm.careunit AS icu_type
    FROM read_parquet('{pm_path}') pm
    LEFT JOIN read_parquet('{p_path}') p ON pm.subject_id = p.subject_id
    LEFT JOIN read_parquet('{hw_path}') hw ON pm.stay_id = hw.stay_id;
    """

    conn.execute(query)

    # Hiển thị thống kê nhanh số lượng bản ghi vừa tạo
    total_rows = conn.execute("SELECT COUNT(*) FROM patient_static_features").fetchone()[0]
    print(f"   -> Đã khởi tạo xong 'patient_static_features' với {total_rows} bản ghi.")

    if 'print_group_summary' in globals() or 'print_group_summary' in locals():
        print_group_summary("1 — Patient Static Features", "patient_static_features")

## **2. Vital Features (vital_features — Multi-Window)**

**Mô tả:** Nhóm đặc trưng động thu thập các chuỗi dữ liệu sinh hiệu dày đặc (High-frequency Time-series) từ chartevents. Thay vì gộp chung toàn bộ quá trình nằm viện, dữ liệu được cắt lát nghiêm ngặt theo các cửa sổ thời gian lũy tiến: $1\text{h}$, $6\text{h}$, $12\text{h}$, và $24\text{h}$ đầu tiên kể từ mốc thời gian bệnh nhân nhập khoa ICU (icu_intime). Phương pháp này giúp mô hình nắm bắt được xu hướng diễn biến cấp tính (Trend) của bệnh nhân mà không gây rò rỉ dữ liệu (Data Leakage).

**Các tính năng tính toán:**

Các chỉ số sinh hiệu cốt lõi bao gồm Nhịp tim (heart_rate), Nhịp thở (resp_rate), Huyết áp trung bình (map_bp), Nhiệt độ cơ thể (temperature) và Độ bão hòa oxy máu ngoại vi (spo2).

Tại mỗi cửa sổ thời gian, các hàm gộp thống kê sẽ được áp dụng độc lập:
- Giá trị trung bình (_avg): Đại diện cho trạng thái nền ổn định của bệnh nhân trong cửa sổ đó (ví dụ: heart_rate_avg_1h, map_bp_avg_6h).
- Giá trị cực tiểu (_min) & Cực đại (_max): Bắt trọn các pha chuyển biến xấu đột ngột hoặc các biến cố sốc, suy hô hấp nguy hiểm (ví dụ: spo2_min_24h, temperature_max_12h).

In [25]:
def build_feature_group_2(silver_dir, conn=None):
    """
    Feature Group 2:
    Xãy dựng bảng vital_features tính toán đa cửa sổ thời gian (1h, 6h, 12h, 24h)
    nghiêm ngặt chống rò rỉ dữ liệu (Anti-Data Leakage).
    """
    print(" -> [FG2] Đang xây dựng: vital_features (Tính toán đa cửa sổ thời gian)...")

    if conn is None:
        conn = get_duckdb_connection()

    # Đường dẫn mẫu nạp dữ liệu phân mảnh (Bucketing/Hive) từ tầng Silver
    vitals_pattern = os.path.join(silver_dir, "clinical", "vital_signs", "bucket=*", "*.parquet")
    pm_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")

    # Kiểm tra sự tồn tại của file trước khi DuckDB thực thi
    if not glob.glob(vitals_pattern, recursive=True) or not glob.glob(pm_path):
        raise FileNotFoundError("[X] Thiếu dữ liệu Silver cần thiết cho cấu phần sinh hiệu (Vital Signs).")

    query = f"""
        CREATE OR REPLACE TABLE vital_features AS
        WITH vitals_with_time AS (
            SELECT
                v.stay_id,
                v.heart_rate,
                v.sys_bp,
                v.dias_bp,
                -- Tính toán hoặc lấy trực tiếp MAP (Mean Arterial Pressure)
                COALESCE(v.map_bp, ROUND((v.sys_bp + 2 * v.dias_bp) / 3.0, 2)) AS map_bp,
                v.resp_rate,
                v.temperature,
                v.spo2,
                -- Tính khoảng cách thời gian chính xác (đơn vị: giờ) từ mốc nhập viện ICU (T0)
                date_diff('hour', CAST(pm.icu_intime AS TIMESTAMP), CAST(v.chart_time AS TIMESTAMP)) AS hours_since_in
            FROM read_parquet('{vitals_pattern}') v
            INNER JOIN read_parquet('{pm_path}') pm ON v.stay_id = pm.stay_id
        )
        SELECT
            stay_id,

            -- CỬA SỔ 1 GIỜ ĐẦU (1h Window)

            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 1 THEN heart_rate END), 2) AS heart_rate_avg_1h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 1 THEN map_bp END), 2) AS map_bp_min_1h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 1 THEN spo2 END), 2) AS spo2_min_1h,

            -- CỬA SỔ 6 GIỜ ĐẦU (6h Window)

            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN heart_rate END), 2) AS heart_rate_avg_6h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN heart_rate END), 2) AS heart_rate_max_6h,
            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN map_bp END), 2) AS map_bp_avg_6h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN map_bp END), 2) AS map_bp_min_6h,
            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN resp_rate END), 2) AS resp_rate_avg_6h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN temperature END), 2) AS temperature_max_6h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN spo2 END), 2) AS spo2_min_6h,

            -- CỬA SỔ 12 GIỜ ĐẦU (12h Window)

            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 12 THEN heart_rate END), 2) AS heart_rate_avg_12h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 12 THEN map_bp END), 2) AS map_bp_min_12h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 12 THEN temperature END), 2) AS temperature_max_12h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 12 THEN spo2 END), 2) AS spo2_min_12h,

            -- CỬA SỔ 24 GIỜ ĐẦU (24h Window)

            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN heart_rate END), 2) AS heart_rate_avg_24h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN heart_rate END), 2) AS heart_rate_max_24h,
            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN map_bp END), 2) AS map_bp_avg_24h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN map_bp END), 2) AS map_bp_min_24h,
            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN resp_rate END), 2) AS resp_rate_avg_24h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN resp_rate END), 2) AS resp_rate_max_24h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN temperature END), 2) AS temperature_max_24h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN spo2 END), 2) AS spo2_min_24h

        FROM vitals_with_time
        GROUP BY stay_id;
    """

    conn.execute(query)

    # Đo lường kết quả đầu ra
    total_rows = conn.execute("SELECT COUNT(*) FROM vital_features").fetchone()[0]
    print(f"   -> Đã hoàn thành 'vital_features' phẳng với {total_rows} ca bệnh.")

    if 'print_group_summary' in globals() or 'print_group_summary' in locals():
        print_group_summary("2 — Vital Features", "vital_features")

## **3. Laboratory Features (lab_features)**

**Mô tả:** Nhóm đặc trưng phản ánh trạng thái nội môi, chức năng tạng và các chỉ số nhiễm trùng của bệnh nhân thông qua dữ liệu xét nghiệm sinh hóa máu từ labevents. Nhóm này cũng được áp dụng kỹ thuật cửa sổ thời gian đồng bộ với sinh hiệu để mô hình hóa sự thay đổi cận lâm sàng theo thời gian thực.

**Các tính năng tính toán:**

Tập trung vào các dấu ấn sinh học nhạy cảm nhất trong hồi sức tích cực bao gồm creatinine (chức năng thận), lactate (tưới máu mô/sốc nhiễm khuẩn), Bạch cầu white_blood_cell (phản ứng viêm/nhiễm trùng), hemoglobin (khả năng vận chuyển oxy/mất máu) và Tiểu cầu platelet (rối loạn đông máu).
- **Giá trị cực đại (_max):** Thường áp dụng cho creatinine và lactate trong vòng $24\text{h}$ đầu để đánh giá mức độ tổn thương tạng nghiêm trọng nhất (ví dụ: `creatinine_max_24h`, `lactate_max_6h`).
- **Giá trị trung bình (_avg) & Cực tiểu (_min):** Theo dõi xu thế ổn định của dòng tế bào máu hoặc các pha tụt giảm tiểu cầu nguy cơ xuất huyết (ví dụ: `wbc_avg_24h`, `platelet_min_24h`).

In [26]:
def build_feature_group_3(silver_dir, conn=None):
    """
    Feature Group 3:
    Xây dựng bảng lab_features tính toán các đặc trưng cận lâm sàng (Labs)
    theo đa cửa sổ thời gian (6h, 12h, 24h) nhằm chống rò rỉ dữ liệu tuyệt đối.
    """
    print(" -> [FG3] Đang xây dựng: lab_features (Tính toán xét nghiệm đa cửa sổ)...")

    if conn is None:
        conn = get_duckdb_connection()

    # Đường dẫn mẫu nạp dữ liệu phân mảnh từ tầng Silver
    labs_pattern = os.path.join(silver_dir, "clinical", "laboratory", "bucket=*", "*.parquet")
    pm_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")

    # Kiểm tra sự tồn tại của file trước khi DuckDB thực thi
    if not glob.glob(labs_pattern, recursive=True) or not glob.glob(pm_path):
        raise FileNotFoundError("[X] Thiếu dữ liệu Silver cần thiết cho cấu phần xét nghiệm (Laboratory).")

    query = f"""
        CREATE OR REPLACE TABLE lab_features AS
        WITH labs_with_time AS (
            SELECT
                l.stay_id,
                l.creatinine,
                l.lactate,
                l.white_blood_cell,
                l.hemoglobin,
                l.platelet,
                -- Tính khoảng cách từ mốc vào ICU (T0) tới thời điểm lấy mẫu xét nghiệm
                date_diff('hour', CAST(pm.icu_intime AS TIMESTAMP), CAST(l.charttime AS TIMESTAMP)) AS hours_since_in
            FROM read_parquet('{labs_pattern}') l
            INNER JOIN read_parquet('{pm_path}') pm ON l.stay_id = pm.stay_id
        )
        SELECT
            stay_id,

            -- CỬA SỔ 6 GIỜ ĐẦU (6h Window) - Tập trung vào dấu hiệu sốc sớm (Lactate)

            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN lactate END), 2) AS lactate_max_6h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 6 THEN creatinine END), 2) AS creatinine_max_6h,

            -- CỬA SỔ 12 GIỜ ĐẦU (12h Window)

            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 12 THEN lactate END), 2) AS lactate_max_12h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 12 THEN creatinine END), 2) AS creatinine_max_12h,
            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 12 THEN white_blood_cell END), 2) AS wbc_avg_12h,

            -- CỬA SỔ 24 GIỜ ĐẦU (24h Window) - Đầy đủ bộ chỉ số nền cho ngày đầu tiên

            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN creatinine END), 2) AS creatinine_max_24h,
            ROUND(MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN lactate END), 2) AS lactate_max_24h,
            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN white_blood_cell END), 2) AS wbc_avg_24h,
            ROUND(MIN(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN platelet END), 2) AS platelet_min_24h,
            ROUND(AVG(CASE WHEN hours_since_in BETWEEN 0 AND 24 THEN hemoglobin END), 2) AS hemoglobin_avg_24h

        FROM labs_with_time
        GROUP BY stay_id;
    """

    conn.execute(query)

    # Thống kê nhanh kết quả
    total_rows = conn.execute("SELECT COUNT(*) FROM lab_features").fetchone()[0]
    print(f"   -> Đã hoàn thành 'lab_features' phẳng với {total_rows} ca bệnh.")

    if 'print_group_summary' in globals() or 'print_group_summary' in locals():
        print_group_summary("3 — Laboratory Features", "lab_features")

## **4. Medication Features (medication_features)**

**Mô tả:**

Nhóm đặc trưng định lượng mức độ can thiệp y tế tích lũy từ bảng `inputevents`. Các chỉ số này phản ánh gián tiếp độ nghiêm trọng của ca bệnh dựa trên lượng thuốc và dịch truyền mà các bác sĩ lâm sàng bắt buộc phải sử dụng để duy trì sự sống cho bệnh nhân trong $24\text{h}$ đầu tiên.

**Các tính năng tính toán:**
- **Tổng lượng dịch truyền `(fluid_total_ml)`:** Tổng thể tích dịch hồi sức đã truyền vào cơ thể tích lũy theo các mốc $6\text{h}$, $12\text{h}$, $24\text{h}$, giúp đánh giá tình trạng quá tải dịch hoặc mức độ bù dịch tích cực.
- **Tổng liều vận mạch `(vasopressor_total_dose)`:** Tổng lượng thuốc vận mạch (ví dụ: Norepinephrine) được sử dụng lũy tiến, là biến số vàng để thuật toán nhận diện sốc nhiễm khuẩn hoặc suy tuần hoàn nén dòng.
- **Tổng lượng Insulin `(insulin_total_units)`:** Tổng liều lượng Insulin sử dụng để kiểm soát đường huyết cấp tính trong ICU.
- **Thời gian can thiệp vận mạch `(vasopressor_duration_hours)`:** Tính toán tổng số giờ bệnh nhân phải phụ thuộc vào các thuốc hỗ trợ co bóp cơ tim và nâng huyết áp trong ngày đầu tiên.

In [27]:
def build_feature_group_4(silver_dir, conn=None):
    """
    Feature Group 4:
    Xây dựng bảng medication_features tính toán tổng liều tích lũy
    và thời gian can thiệp dược lý theo các cửa sổ (6h, 12h, 24h).
    """
    print(" -> [FG4] Đang xây dựng: medication_features (Tính toán liều can thiệp tích lũy)...")

    if conn is None:
        conn = get_duckdb_connection()

    # Đường dẫn nạp dữ liệu từ tầng Silver
    meds_pattern = os.path.join(silver_dir, "clinical", "medication", "*.parquet")
    pm_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")

    # Kiểm tra sự tồn tại của file trước khi thực thi
    if not glob.glob(meds_pattern) or not glob.glob(pm_path):
        raise FileNotFoundError("[X] Thiếu dữ liệu Silver cần thiết cho cấu phần thuốc (Medication).")

    query = f"""
        CREATE OR REPLACE TABLE medication_features AS
        WITH meds_with_time AS (
            SELECT
                m.stay_id,
                m.med_name,
                m.dose,
                -- Tính khoảng cách từ lúc vào ICU tới khi bắt đầu dùng liều thuốc này
                date_diff('hour', CAST(pm.icu_intime AS TIMESTAMP), CAST(m.starttime AS TIMESTAMP)) AS hours_since_in,
                -- Tính thời gian sử dụng của liều này (đơn vị: giờ) phục vụ tính Vasopressor Duration
                date_diff('hour', CAST(m.starttime AS TIMESTAMP), CAST(m.endtime AS TIMESTAMP)) AS dose_duration_hours
            FROM read_parquet('{meds_pattern}') m
            INNER JOIN read_parquet('{pm_path}') pm ON m.stay_id = pm.stay_id
        )
        SELECT
            stay_id,

            -- =========================================================================
            -- CỬA SỔ 6 GIỜ ĐẦU (6h Window)
            -- =========================================================================
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 6 AND (med_name LIKE '%Fluid%' OR med_name LIKE '%NaCl%') THEN dose END), 2) AS fluid_total_ml_6h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 6 AND (med_name LIKE '%Norepinephrine%' OR med_name LIKE '%Epinephrine%') THEN dose END), 2) AS vaso_total_dose_6h,

            -- =========================================================================
            -- CỬA SỔ 12 GIỜ ĐẦU (12h Window)
            -- =========================================================================
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND (med_name LIKE '%Fluid%' OR med_name LIKE '%NaCl%') THEN dose END), 2) AS fluid_total_ml_12h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND (med_name LIKE '%Norepinephrine%' OR med_name LIKE '%Epinephrine%') THEN dose END), 2) AS vaso_total_dose_12h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND med_name LIKE '%Insulin%' THEN dose END), 2) AS insulin_total_units_12h,

            -- =========================================================================
            -- CỬA SỔ 24 GIỜ ĐẦU (24h Window)
            -- =========================================================================
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND (med_name LIKE '%Fluid%' OR med_name LIKE '%NaCl%') THEN dose END), 2) AS fluid_total_ml_24h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND (med_name LIKE '%Norepinephrine%' OR med_name LIKE '%Epinephrine%') THEN dose END), 2) AS vaso_total_dose_24h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND med_name LIKE '%Insulin%' THEN dose END), 2) AS insulin_total_units_24h,

            -- Tính tổng số giờ phải duy trì vận mạch trong ngày đầu tiên (Vasopressor Duration)
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND (med_name LIKE '%Norepinephrine%' OR med_name LIKE '%Epinephrine%') THEN dose_duration_hours END), 2) AS vaso_duration_hours_24h

        FROM meds_with_time
        GROUP BY stay_id;
    """

    conn.execute(query)

    # Thống kê nhanh kết quả
    total_rows = conn.execute("SELECT COUNT(*) FROM medication_features").fetchone()[0]
    print(f"   ✓ Đã hoàn thành 'medication_features' phẳng với {total_rows} ca bệnh.")

    if 'print_group_summary' in globals() or 'print_group_summary' in locals():
        print_group_summary("4 — Medication Features", "medication_features")

## **5. Fluid Outputs Features (fluid_output_features — Cumulative Window)**

**Mô tả:**

Nhóm đặc trưng động theo dõi thể tích dịch bài tiết và các nguồn dịch ra từ cơ thể bệnh nhân được khai thác từ `outputevents`. Đây là nhóm chỉ số cực kỳ nhạy bén giúp mô hình AI kết hợp với dữ liệu dịch vào (từ medication_features) để tính toán cân bằng dịch (Fluid Balance), đồng thời là tiêu chuẩn vàng để đánh giá chức năng bài tiết của hệ tiết niệu theo thời gian thực.

**Các tính năng tính toán:**

Tập trung chủ yếu vào thể tích nước tiểu (Urine Output) và các dịch dẫn lưu khác, được cộng dồn tích lũy theo các mốc cửa sổ thời gian lũy tiến: $1\text{h}$, $6\text{h}$, $12\text{h}$, và $24\text{h}$ đầu tiên kể từ khi vào ICU.
- **Tổng thể tích nước tiểu tích lũy (`urine_output_sum_1h, urine_output_sum_6h`,...):** Tổng lượng ml nước tiểu thu được trong từng cửa sổ.
-> Chỉ số này khi giảm sâu (ví dụ: $< 0.5\text{ ml/kg/giờ}$ trong cửa sổ $6\text{h}$) là dấu hiệu cảnh báo sớm của suy thận cấp (AKI) hoặc tình trạng giảm tưới máu thận do sốc.

In [30]:
def build_feature_group_5(silver_dir, conn=None):
    """
    Feature Group 5:
    Xây dựng bảng output_features tính toán tổng lượng dịch ra tích lũy (Urine, Drains)
    theo các cửa sổ thời gian (1h, 6h, 12h, 24h) phục vụ đánh giá chức năng thận và cân bằng dịch.
    """
    print(" -> [FG5] Đang xây dựng: output_features (Tính toán dịch ra tích lũy)...")

    if conn is None:
        conn = get_duckdb_connection()

    # Đường dẫn nạp dữ liệu từ tầng Silver
    outputs_pattern = os.path.join(silver_dir, "clinical", "outputs", "**", "*.parquet")
    pm_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")

    # Kiểm tra sự tồn tại của file trước khi thực thi
    if not glob.glob(outputs_pattern, recursive=True) or not glob.glob(pm_path):
        raise FileNotFoundError("[X] Thiếu dữ liệu Silver cần thiết cho cấu phần dịch ra (Outputs).")

    query = f"""
        CREATE OR REPLACE TABLE output_features AS
        WITH outputs_with_time AS (
            SELECT
                o.stay_id,
                o.output_type,
                o.volume_ml,
                -- Tính khoảng cách từ lúc vào ICU tới khi ghi nhận lượng dịch ra này
                date_diff('hour', CAST(pm.icu_intime AS TIMESTAMP), CAST(o.charttime AS TIMESTAMP)) AS hours_since_in
            FROM read_parquet('{outputs_pattern}', hive_partitioning=1) o
            INNER JOIN read_parquet('{pm_path}') pm ON o.stay_id = pm.stay_id
        )
        SELECT
            stay_id,

            -- CỬA SỔ 1 GIỜ ĐẦU (1h Window)

            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 1 AND output_type LIKE '%Urine%' THEN volume_ml END), 2) AS urine_output_sum_1h,

            -- CỬA SỔ 6 GIỜ ĐẦU (6h Window) - Mốc cực kỳ quan trọng để đánh giá thiểu niệu KDIGO Stage 1

            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 6 AND output_type LIKE '%Urine%' THEN volume_ml END), 2) AS urine_output_sum_6h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 6 AND output_type LIKE '%Drain%' THEN volume_ml END), 2) AS drain_output_sum_6h,

            -- CỬA SỔ 12 GIỜ ĐẦU (12h Window) - Mốc đánh giá thiểu niệu tiến triển KDIGO Stage 2

            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND output_type LIKE '%Urine%' THEN volume_ml END), 2) AS urine_output_sum_12h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND output_type LIKE '%Drain%' THEN volume_ml END), 2) AS drain_output_sum_12h,

            -- CỬA SỔ 24 GIỜ ĐẦU (24h Window) - Toàn cảnh ngày đầu tiên tại ICU

            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND output_type LIKE '%Urine%' THEN volume_ml END), 2) AS urine_output_sum_24h,
            ROUND(SUM(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND output_type LIKE '%Drain%' THEN volume_ml END), 2) AS drain_output_sum_24h

        FROM outputs_with_time
        GROUP BY stay_id;
    """

    conn.execute(query)

    # Thống kê nhanh kết quả
    total_rows = conn.execute("SELECT COUNT(*) FROM output_features").fetchone()[0]
    print(f"   -> Đã hoàn thành 'output_features' phẳng với {total_rows} ca bệnh.")

    if 'print_group_summary' in globals() or 'print_group_summary' in locals():
        print_group_summary("5 — Output Features", "output_features")

## **6. Target Labels (target_labels — One-to-Many Mapping)**
**Mô tả:**

Nhóm nhãn mục tiêu (Ground Truth) được thiết kế để lưu trữ hoàn toàn độc lập với Ma trận Đặc trưng (Features). Các nhãn này được xây dựng dựa trên các định nghĩa luật y khoa (Rule-based Clinical Definition) hoặc các biến cố thực tế xảy ra trong suốt quá trình nằm viện. Cấu trúc tách biệt này tạo nên kiến trúc Một Feature Store — Nhiều Nhãn Đầu Ra, cho phép các mô hình AI khác nhau liên kết trực tiếp để huấn luyện mà không cần chạy lại pipeline dữ liệu thô.

**Các tính năng tính toán (Các nhãn đầu ra chính):**

- **Nhãn Tử vong nội viện `(label_mortality)`:** Nhãn nhị phân ($0/1$), xác định xem bệnh nhân có tử vong trong đợt điều trị ICU hoặc trước thời điểm xuất viện hay không.
- **Nhãn Suy thận cấp `(label_aki_stage)`:** Nhãn phân loại đa lớp ($0, 1, 2, 3$) được chấm điểm tự động theo tiêu chuẩn lâm sàng KDIGO quốc tế, dựa trên tỷ lệ tăng vọt của Creatinine trong $24\text{h}$ đầu so với mức nền và mức độ thiểu niệu từ nhóm fluid_output_features.
- **Nhãn Nhiễm trùng huyết `(label_sepsis)`:** Nhãn nhị phân ($0/1$), đóng vai trò là một Proxy lâm sàng chuẩn Sepsis-3 khi ghi nhận bệnh nhân xuất hiện tình trạng nhiễm trùng đi kèm suy tuần hoàn cấp (phải sử dụng thuốc vận mạch liều cao).
- **Nhãn Nằm viện kéo dài `(label_long_los)`:** Nhãn nhị phân ($0/1$), gán giá trị $1$ nếu thời gian nằm viện thực tế tại ICU của bệnh nhân kéo dài vượt quá mốc $3$ ngày, phục vụ bài toán tối ưu hóa và dự báo quá tải hạ tầng giường bệnh.

In [31]:
def build_feature_group_6(silver_dir, conn=None):
    """
    Feature Group 6:
    Xây dựng bảng procedure_features chuyển đổi các can thiệp y tế xâm lấn
    (Thở máy, Lọc máu, ECMO, Đặt nội khí quản) thành các biến Flag (0/1)
    theo các cửa sổ thời gian (6h, 12h, 24h) nhằm chống rò rỉ dữ liệu.
    """
    print(" -> [FG6] Đang xây dựng: procedure_features (Tính toán can thiệp lâm sàng)...")

    if conn is None:
        conn = get_duckdb_connection()

    # Đường dẫn nạp dữ liệu từ tầng Silver
    procedures_pattern = os.path.join(silver_dir, "clinical", "procedures", "*.parquet")
    pm_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")

    # Kiểm tra sự tồn tại của file trước khi thực thi
    if not glob.glob(procedures_pattern) or not glob.glob(pm_path):
        raise FileNotFoundError("[X] Thiếu dữ liệu Silver cần thiết cho cấu phần thủ thuật (Procedures).")

    query = f"""
        CREATE OR REPLACE TABLE procedure_features AS
        WITH procedures_with_time AS (
            SELECT
                proc.stay_id,
                proc.procedure_name,
                -- Tính khoảng cách từ lúc vào ICU tới khi bắt đầu thực hiện thủ thuật
                -- Sử dụng COALESCE để phòng ngừa trường hợp bảng thủ thuật dùng 'starttime' hoặc 'charttime'
                date_diff(
                    'hour',
                    CAST(pm.icu_intime AS TIMESTAMP),
                    CAST(COALESCE(proc.starttime, proc.charttime) AS TIMESTAMP)
                ) AS hours_since_in
            FROM read_parquet('{procedures_pattern}') proc
            INNER JOIN read_parquet('{pm_path}') pm ON proc.stay_id = pm.stay_id
        )
        SELECT
            stay_id,

            -- CỬA SỔ 6 GIỜ ĐẦU (6h Window)

            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 6 AND procedure_name LIKE '%Ventilation%' THEN 1 END) AS mech_vent_6h,
            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 6 AND (procedure_name LIKE '%Dialysis%' OR procedure_name LIKE '%CRRT%') THEN 1 END) AS dialysis_6h,
            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 6 AND procedure_name LIKE '%Intubation%' THEN 1 END) AS intubation_6h,

            -- CỬA SỔ 12 GIỜ ĐẦU (12h Window)

            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND procedure_name LIKE '%Ventilation%' THEN 1 END) AS mech_vent_12h,
            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND (procedure_name LIKE '%Dialysis%' OR procedure_name LIKE '%CRRT%') THEN 1 END) AS dialysis_12h,
            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 12 AND procedure_name LIKE '%Intubation%' THEN 1 END) AS intubation_12h,

            -- CỬA SỔ 24 GIỜ ĐẦU (24h Window)

            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND procedure_name LIKE '%Ventilation%' THEN 1 END) AS mech_vent_24h,
            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND (procedure_name LIKE '%Dialysis%' OR procedure_name LIKE '%CRRT%') THEN 1 END) AS dialysis_24h,
            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND procedure_name LIKE '%ECMO%' THEN 1 END) AS ecmo_24h,
            MAX(CASE WHEN hours_since_in BETWEEN 0 AND 24 AND procedure_name LIKE '%Intubation%' THEN 1 END) AS intubation_24h

        FROM procedures_with_time
        GROUP BY stay_id;
    """

    conn.execute(query)

    # Thống kê nhanh kết quả
    total_rows = conn.execute("SELECT COUNT(*) FROM procedure_features").fetchone()[0]
    print(f"   ✓ Đã hoàn thành 'procedure_features' phẳng với {total_rows} ca bệnh.")

    if 'print_group_summary' in globals() or 'print_group_summary' in locals():
        print_group_summary("6 — Procedure Features", "procedure_features")

### **Hợp nhất thành Unified Patient Feature Store (patient_feature_store)**

Toàn bộ 6 nhóm đặc trưng (Feature Groups) sau khi được tính toán độc lập tại tầng Gold sẽ trải qua một bước hợp nhất vật lý (Physical Flattening). Phép toán `LEFT JOIN` thông qua khóa chính stay_id (mã định danh duy nhất cho mỗi ca nằm viện ICU) sẽ liên kết toàn bộ các chiều không gian đặc trưng tĩnh và động để tạo ra một Ma trận Đặc trưng phẳng duy nhất (Unified Feature Matrix).$$\text{Patient Master} \xrightarrow{\text{LEFT JOIN}} \begin{cases} \text{Static Features} \\ \text{Vital Features (Multi-window)} \\ \text{Laboratory Features} \\ \text{Medication Features} \\ \text{Output Features} \\ \text{Procedure Features} \end{cases} \xrightarrow{\text{FLATTEN}} \mathbf{Patient\ Feature\ Store}$$
- **Tư duy xử lý dữ liệu khuyết khuyết (Clinical ML Imputation Logic):** Trong môi trường hồi sức tích cực (ICU), việc xử lý giá trị khuyết (NULL) sau khi LEFT JOIN đòi hỏi sự am hiểu sâu sắc về mặt lâm sàng:
  - **Đối với các can thiệp (Meds, Outputs, Procedures):** Nếu bệnh nhân mang giá trị `NULL` sau khi ghép bảng, điều đó có nghĩa là họ không thực hiện can thiệp đó. Do đó, các trường này sẽ được chuyển đổi (COALESCE) về 0 một cách an toàn (ví dụ: không thở máy = 0, liều vận mạch = 0).
  - **Đối với dấu hiệu sinh tồn và xét nghiệm (Vitals, Labs):** Nếu mang giá trị NULL, đó là do thiếu phép đo (Missing measurements).
  
  Chúng ta tuyệt đối không tự ý điền 0 (vì Nhịp tim = 0 hay Creatinine = 0 mang ý nghĩa sinh học cực đoan). Các giá trị này sẽ được giữ nguyên là NULL để các thuật toán Gradient Boosting (XGBoost, LightGBM) tự học phân phối khuyết hoặc để kỹ sư Data Science xử lý bằng các kỹ thuật Imputation (MICE, KNN Imputer) ở tầng Train.

In [32]:
def build_unified_feature_store(gold_dir, conn=None):
    """
    Hợp nhất 6 Feature Groups thành Unified Patient Feature Store duy nhất.
    - Áp dụng kỹ thuật COALESCE lâm sàng chuẩn hóa cho các can thiệp (0/1, liều lượng).
    - Giữ nguyên NULL tự nhiên cho các phép đo lâm sàng (Vitals, Labs).
    - Phân mảnh vật lý thành 8 buckets Parquet nhằm tăng tốc độ truy xuất khi huấn luyện.
    """
    print("\n=== [FINAL] Bắt đầu hợp nhất dữ liệu thành Unified Feature Store ===")

    if conn is None:
        conn = get_duckdb_connection()

    # Thư mục đích lưu trữ Feature Store
    features_output_path = os.path.join(gold_dir, "features")
    os.makedirs(features_output_path, exist_ok=True)

    # Truy vấn hợp nhất 6 bảng đặc trưng logic
    query = f"""
        CREATE OR REPLACE TABLE patient_feature_store AS
        SELECT
            -- 1. Base Static Features
            sf.stay_id,
            sf.subject_id,
            sf.hadm_id,
            sf.age,
            sf.gender,
            sf.height_cm,
            sf.weight_kg,
            sf.bmi,
            sf.admission_type,
            sf.icu_type,

            -- 2. Vital Signs (Giữ nguyên NULL nếu không đo đạc)
            vf.* EXCLUDE (stay_id),

            -- 3. Laboratories (Giữ nguyên NULL nếu không xét nghiệm)
            lf.* EXCLUDE (stay_id),

            -- 4. Medications (COALESCE về 0 nếu không sử dụng thuốc/dịch truyền)
            COALESCE(mf.fluid_total_ml_6h, 0.0) AS fluid_total_ml_6h,
            COALESCE(mf.vaso_total_dose_6h, 0.0) AS vaso_total_dose_6h,
            COALESCE(mf.fluid_total_ml_12h, 0.0) AS fluid_total_ml_12h,
            COALESCE(mf.vaso_total_dose_12h, 0.0) AS vaso_total_dose_12h,
            COALESCE(mf.insulin_total_units_12h, 0.0) AS insulin_total_units_12h,
            COALESCE(mf.fluid_total_ml_24h, 0.0) AS fluid_total_ml_24h,
            COALESCE(mf.vaso_total_dose_24h, 0.0) AS vaso_total_dose_24h,
            COALESCE(mf.insulin_total_units_24h, 0.0) AS insulin_total_units_24h,
            COALESCE(mf.vaso_duration_hours_24h, 0.0) AS vaso_duration_hours_24h,

            -- 5. Fluid Outputs (COALESCE về 0 nếu không ghi nhận dịch bài tiết)
            COALESCE(of.urine_output_sum_1h, 0.0) AS urine_output_sum_1h,
            COALESCE(of.urine_output_sum_6h, 0.0) AS urine_output_sum_6h,
            COALESCE(of.drain_output_sum_6h, 0.0) AS drain_output_sum_6h,
            COALESCE(of.urine_output_sum_12h, 0.0) AS urine_output_sum_12h,
            COALESCE(of.drain_output_sum_12h, 0.0) AS drain_output_sum_12h,
            COALESCE(of.urine_output_sum_24h, 0.0) AS urine_output_sum_24h,
            COALESCE(of.drain_output_sum_24h, 0.0) AS drain_output_sum_24h,

            -- 6. Procedures (COALESCE về 0 nếu không thực hiện thủ thuật)
            COALESCE(pf.mech_vent_6h, 0) AS mech_vent_6h,
            COALESCE(pf.dialysis_6h, 0) AS dialysis_6h,
            COALESCE(pf.intubation_6h, 0) AS intubation_6h,
            COALESCE(pf.mech_vent_12h, 0) AS mech_vent_12h,
            COALESCE(pf.dialysis_12h, 0) AS dialysis_12h,
            COALESCE(pf.intubation_12h, 0) AS intubation_12h,
            COALESCE(pf.mech_vent_24h, 0) AS mech_vent_24h,
            COALESCE(pf.dialysis_24h, 0) AS dialysis_24h,
            COALESCE(pf.ecmo_24h, 0) AS ecmo_24h,
            COALESCE(pf.intubation_24h, 0) AS intubation_24h,

            -- Kỹ thuật băm dữ liệu (Bucketing) phân mảnh lưu trữ
            ABS(HASH(sf.stay_id)) % 8 AS bucket_id

        FROM patient_static_features sf
        LEFT JOIN vital_features vf ON sf.stay_id = vf.stay_id
        LEFT JOIN lab_features lf ON sf.stay_id = lf.stay_id
        LEFT JOIN medication_features mf ON sf.stay_id = mf.stay_id
        LEFT JOIN output_features of ON sf.stay_id = of.stay_id
        LEFT JOIN procedure_features pf ON sf.stay_id = pf.stay_id;
    """
    conn.execute(query)
    print("   ✓ Đã thực hiện liên kết phẳng (Flattening) 6 Feature Groups thành công.")

    # Xuất bản ma trận đặc trưng ra thư mục Gold dưới dạng Parquet phân mảnh
    export_query = f"""
        COPY patient_feature_store
        TO '{features_output_path}'
        (FORMAT PARQUET, PARTITION_BY bucket_id, OVERWRITE_OR_IGNORE 1);
    """
    conn.execute(export_query)

    # Đo lường thống kê đầu ra
    total_patients = conn.execute("SELECT COUNT(DISTINCT stay_id) FROM patient_feature_store").fetchone()[0]
    print(f"   ✓ Unified Feature Store đã xuất xưởng thành công!")
    print(f"   ✓ Vị trí lưu trữ: {features_output_path}/")
    print(f"   ✓ Tổng số lượng thực thể lâm sàng (stay_id): {total_patients} bệnh nhân.")
    print("==========================================================================\n")